In [69]:
import pandas as pd
import numpy as np

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

# The data is interleaved with features on one line and the target on the next.
# We need to separate them.
data = raw_df.values[::2, :]
target = raw_df.values[1::2, 2] # The target is in the 3rd column (index 2) of the second row

# The last two columns of the data are combined in the raw data, so we need to split them
data = np.hstack([data[:, :9], data[:, 9:11]])

# Convert data and target to float, coercing errors
data = data.astype(float)
target = target.astype(float)

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-1249776968.py:5: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


In [70]:
# Check for and remove rows with NaN in y
nan_mask = np.isnan(target)
X = data[~nan_mask]
y = target[~nan_mask]

In [71]:
X.shape

(506, 11)

In [72]:
y.shape

(506,)

In [73]:
X

array([[6.3200e-03, 1.8000e+01, 2.3100e+00, ..., 1.0000e+00, 2.9600e+02,
        1.5300e+01],
       [2.7310e-02, 0.0000e+00, 7.0700e+00, ..., 2.0000e+00, 2.4200e+02,
        1.7800e+01],
       [2.7290e-02, 0.0000e+00, 7.0700e+00, ..., 2.0000e+00, 2.4200e+02,
        1.7800e+01],
       ...,
       [6.0760e-02, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01],
       [1.0959e-01, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01],
       [4.7410e-02, 0.0000e+00, 1.1930e+01, ..., 1.0000e+00, 2.7300e+02,
        2.1000e+01]])

In [74]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score

In [75]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR()

In [76]:
estimators = [('lr',lr),('dt',dt),('svr',svr)]

In [77]:
for estimator in estimators:
    scores = cross_val_score(estimator[1],X,y,cv=10,scoring='r2')
    print(estimator[0],np.round(np.mean(scores),2))

lr 0.15
dt -2.67
svr -0.42


In [78]:
print("Shape of data before NaN removal:", data.shape)
print("Number of NaNs in data before NaN removal:", np.isnan(data).sum())
print("Shape of target before NaN removal:", target.shape)
print("Number of NaNs in target before NaN removal:", np.isnan(target).sum())

nan_mask = np.isnan(target)
X = data[~nan_mask]
y = target[~nan_mask]

print("Shape of X after NaN removal:", X.shape)
print("Shape of y after NaN removal:", y.shape)

Shape of data before NaN removal: (506, 11)
Number of NaNs in data before NaN removal: 0
Shape of target before NaN removal: (506,)
Number of NaNs in target before NaN removal: 0
Shape of X after NaN removal: (506, 11)
Shape of y after NaN removal: (506,)


In [79]:
from sklearn.ensemble import VotingRegressor

In [80]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,cv=10,scoring='r2')
print(np.round(np.mean(scores),2))

0.26


In [81]:
dt1 = DecisionTreeRegressor(max_depth=1)
dt2 = DecisionTreeRegressor(max_depth=3)
dt3 = DecisionTreeRegressor(max_depth=5)
dt4 = DecisionTreeRegressor(max_depth=7)
dt5 = DecisionTreeRegressor(max_depth=None)

estimators = [('dt1',dt1),('dt2',dt2),('dt3',dt3),('dt4',dt4),('dt5',dt5)]

# Iterate through combinations of 5 weights, each ranging from 1 to 3
for w1 in range(1, 4):
  for w2 in range(1, 4):
    for w3 in range(1, 4):
      for w4 in range(1, 4):
        for w5 in range(1, 4):
          weights = [w1, w2, w3, w4, w5]
          vr = VotingRegressor(estimators, weights=weights)
          scores = cross_val_score(vr, X, y, cv=10, scoring='r2')
          print("for weights = {}, R2 score = {}".format(weights, np.round(np.mean(scores), 2)))

for weights = [1, 1, 1, 1, 1], R2 score = 0.02
for weights = [1, 1, 1, 1, 2], R2 score = 0.1
for weights = [1, 1, 1, 1, 3], R2 score = -0.54
for weights = [1, 1, 1, 2, 1], R2 score = -0.19
for weights = [1, 1, 1, 2, 2], R2 score = -0.36
for weights = [1, 1, 1, 2, 3], R2 score = -0.56
for weights = [1, 1, 1, 3, 1], R2 score = -0.49
for weights = [1, 1, 1, 3, 2], R2 score = -0.52
for weights = [1, 1, 1, 3, 3], R2 score = -0.68
for weights = [1, 1, 2, 1, 1], R2 score = -0.05
for weights = [1, 1, 2, 1, 2], R2 score = -0.2
for weights = [1, 1, 2, 1, 3], R2 score = 0.17
for weights = [1, 1, 2, 2, 1], R2 score = -0.18
for weights = [1, 1, 2, 2, 2], R2 score = 0.08
for weights = [1, 1, 2, 2, 3], R2 score = -0.4
for weights = [1, 1, 2, 3, 1], R2 score = -0.22
for weights = [1, 1, 2, 3, 2], R2 score = -0.12
for weights = [1, 1, 2, 3, 3], R2 score = -0.05
for weights = [1, 1, 3, 1, 1], R2 score = -0.18
for weights = [1, 1, 3, 1, 2], R2 score = -0.03
for weights = [1, 1, 3, 1, 3], R2 score = -0.47

In [82]:
dt1 = DecisionTreeRegressor(max_depth=1)
dt2 = DecisionTreeRegressor(max_depth=3)
dt3 = DecisionTreeRegressor(max_depth=5)
dt4 = DecisionTreeRegressor(max_depth=7)
dt5 = DecisionTreeRegressor(max_depth=None)

In [83]:
estimators = [('dt1',dt1),('dt2',dt2),('dt3',dt3),('dt4',dt4),('dt5',dt5)]

In [84]:
print("Number of NaNs in X before cross-validation:", np.isnan(X).sum())
print("Number of NaNs in y before cross-validation:", np.isnan(y).sum())

# Perform cross-validation on each estimator individually
scores_dt1 = cross_val_score(estimators[0][1], X, y, cv=10, scoring='r2')
print(estimators[0][0], np.round(np.mean(scores_dt1), 2))

scores_dt2 = cross_val_score(estimators[1][1], X, y, cv=10, scoring='r2')
print(estimators[1][0], np.round(np.mean(scores_dt2), 2))

scores_dt3 = cross_val_score(estimators[2][1], X, y, cv=10, scoring='r2')
print(estimators[2][0], np.round(np.mean(scores_dt3), 2))

scores_dt4 = cross_val_score(estimators[3][1], X, y, cv=10, scoring='r2')
print(estimators[3][0], np.round(np.mean(scores_dt4), 2))

scores_dt5 = cross_val_score(estimators[4][1], X, y, cv=10, scoring='r2')
print(estimators[4][0], np.round(np.mean(scores_dt5), 2))

Number of NaNs in X before cross-validation: 0
Number of NaNs in y before cross-validation: 0
dt1 -0.37
dt2 0.05
dt3 0.11
dt4 -1.04
dt5 -2.85


In [85]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr,X,y,cv=10,scoring='r2')
print(np.round(np.mean(scores),2))

0.06


In [86]:
scores = cross_val_score(dt1, X, y, cv=10, scoring='r2')
print("Scores for dt1:", np.round(np.mean(scores), 2))

Scores for dt1: -0.37
